In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import h5py, os, tqdm
os.environ['XLA_PYTHON_CLIENT_MEM_FRACTION'] = '0.49'
import numpy as np
import matplotlib.pyplot as plt
from functools import partial

import jax
import jax.numpy as jnp
import jax_cosmo as jc

from flax import nnx
import orbax.checkpoint as ocp

import diffrax
from diffrax import diffeqsolve, ODETerm, LeapfrogMidpoint, PIDController, SaveAt, ConstantStepSize

import jaxpm
from jaxpm.painting import cic_paint, cic_read
from jaxpm.kernels import fftk, invnabla_kernel
from jaxpm.nn import MLP
from jaxpm import camels, plotting, hpm, nn, data

In [3]:
parts_per_dim = 64
mesh_per_dim = parts_per_dim
mesh_shape = [mesh_per_dim] * 3
box_size = [float(mesh_per_dim)] * 3

In [4]:
out_dict = camels.load_CV_snapshots(
    "CV_1",
    # "CV_0",
    mesh_per_dim,
    parts_per_dim,
    i_snapshots=range(1, 33+4, 4),
    # i_snapshots=np.arange(-4, 0, dtype=int),
    return_hydro=True,
)

cosmo = out_dict["cosmo"]
scales = out_dict["scales"]

dm_poss = out_dict["dm_poss"]
dm_vels = out_dict["dm_vels"]

gas_poss = out_dict["gas_poss"]
gas_vels = out_dict["gas_vels"]

No matching catalogs found, returning only the snapshots
Using snapshots ['/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_018.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_034.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_042.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_050.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_058.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_066.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_074.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_082.hdf5', '/cluster/work/refregier/athomsen/flatiron/CAMELS/Sims/SIMBA/CV/CV_1/snapshot_090.hdf5']
Selecting 262144 dark matter (deterministic)
Selecting 262144 gas particles (random)


finding unique gas particle indices:  22%|██▏       | 2/9 [00:06<00:23,  3.43s/it]

Found 66 duplicate gas particle IDs


finding unique gas particle indices:  33%|███▎      | 3/9 [00:11<00:26,  4.41s/it]

Found 279 duplicate gas particle IDs


...

Found 14533 duplicate gas particle IDs


finding unique gas particle indices: 100%|██████████| 9/9 [00:45<00:00,  5.04s/it]


There are 15868219 (94.58%) gas particles that exist in all snapshots


loading snapshots:   0%|          | 0/9 [00:00<?, ?it/s]2025-04-23 16:19:41.139295: W external/tsl/tsl/framework/bfc_allocator.cc:482] Allocator (GPU_0_bfc) ran out of memory trying to allocate 1.50GiB (rounded to 1608685568)requested by op 
2025-04-23 16:19:41.139456: W external/tsl/tsl/framework/bfc_allocator.cc:494] ******************************************************************************______________________
E0423 16:19:41.139554 3611228 pjrt_stream_executor_client.cc:2826] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 1608685536 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:    1.50GiB
              constant allocation:         0B
        maybe_live_out allocation:    1.50GiB
     preallocated temp allocation:       259B
  preallocated temp fragmentation:       116B (44.79%)
                 total allocation:    3.00GiB
              total fragmentation:       244B (0.00%)
Peak buf

XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 1608685536 bytes.
BufferAssignment OOM Debugging.
BufferAssignment stats:
             parameter allocation:    1.50GiB
              constant allocation:         0B
        maybe_live_out allocation:    1.50GiB
     preallocated temp allocation:       259B
  preallocated temp fragmentation:       116B (44.79%)
                 total allocation:    3.00GiB
              total fragmentation:       244B (0.00%)
Peak buffers:
	Buffer 1:
		Size: 1.50GiB
		Entry Parameter Subshape: s32[16757141,8,3]
		==========================

	Buffer 2:
		Size: 1.50GiB
		Operator: op_name="jit(remainder)/jit(main)/select_n" source_file="/cluster/home/athomsen/flatiron/repos/JaxPM/jaxpm/painting.py" source_line=36
		XLA Label: fusion
		Shape: s32[16757141,8,3]
		==========================

	Buffer 3:
		Size: 16B
		Operator: op_name="jit(remainder)/jit(main)/lt" source_file="/cluster/home/athomsen/flatiron/repos/JaxPM/jaxpm/paintin

In [ ]:
def restore_model(checkpoint_file):
    model = MLP(
        d_in=5,
        d_out=1, 
        d_hidden=64, 
        n_hidden=4, 
        dropout_rate=0.0,
        rngs=nnx.Rngs(0),
        norm_type="batch",
    )
    
    checkpointer = ocp.StandardCheckpointer()
    
    abstract_model = nnx.eval_shape(lambda: model)
    graphdef, abstract_params = nnx.split(abstract_model)
    
    params = checkpointer.restore(checkpoint_file, abstract_params)
    model = nnx.merge(graphdef, params)

    return model
    

In [ ]:
# checkpoint_file = "/cluster/home/athomsen/flatiron/repos/JaxPM/dev/hpm/offline_regression/checkpoints/mlp_table_cv1.jx"
# checkpoint_file = "/cluster/home/athomsen/flatiron/repos/JaxPM/dev/hpm/offline_regression/checkpoints/mlp_table_v2.jx"
checkpoint_file = "/cluster/home/athomsen/flatiron/repos/JaxPM/dev/hpm/offline_regression/checkpoints/mlp_table_v3.jx"

offline_model = restore_model(checkpoint_file)

In [ ]:
# checkpoint_file = "/cluster/home/athomsen/flatiron/repos/JaxPM/dev/hpm/sims/checkpoints/mlp_sim_v2.jx"
# checkpoint_file = "/cluster/home/athomsen/flatiron/repos/JaxPM/dev/hpm/sims/checkpoints/mlp_sim_v3_late.jx"
# checkpoint_file = "/cluster/home/athomsen/flatiron/repos/JaxPM/dev/hpm/sims/checkpoints/mlp_sim_v3_full_weighted.jx"
checkpoint_file = "/cluster/home/athomsen/flatiron/repos/JaxPM/dev/hpm/sims/checkpoints/mlp_sim_v4_full.jx"

online_model = restore_model(checkpoint_file)

# simulation

In [ ]:
# uncorrected reference
og_ode = hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo)
og_res = diffeqsolve(
        terms=ODETerm(og_ode),
        solver=LeapfrogMidpoint(),
        t0=scales[0],
        t1=scales[-1],
        dt0=0.01,
        y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
        saveat=SaveAt(ts=scales),
        max_steps=100,
        stepsize_controller=ConstantStepSize(),
)
og_dm_poss, og_dm_vels, og_gas_poss, og_gas_vels = og_res.ys

# including network correction
nn_ode = hpm.get_hpm_network_ode_fn(mesh_per_dim, cosmo, pressure_model=online_model, gas_architecture="mlp")
nn_res = diffeqsolve(
        terms=ODETerm(nn_ode),
        solver=LeapfrogMidpoint(),
        t0=scales[0],
        t1=scales[-1],
        dt0=0.01,
        y0=(dm_poss[0], dm_vels[0], gas_poss[0], gas_vels[0]),
        saveat=SaveAt(ts=scales),
        max_steps=100,
        stepsize_controller=ConstantStepSize(),
)
nn_dm_poss, nn_dm_vels, nn_gas_poss, nn_gas_vels = nn_res.ys

with jax.default_device(jax.devices("cpu")[0]):
    plotting.compare_particle_evolution(
        mesh_shape, 
        scales, 
        jnp.stack([gas_poss, og_gas_poss, nn_gas_poss], axis=0), 
        title="gas",
        col_titles=["CAMELS", "gravity", "gravity + pressure"],
        include_pk=True,
        include_reference=True,
    )

# tests

In [ ]:
x_labels = ["rho", "fscalar", "vel_disp", "vel_div"]
# x_labels = ["rho", "fscalar"]
y_labels = ["P"]

X, _, Y, _ = data.get_offline_regression_data(
    out_dict, x_labels=x_labels, y_labels=y_labels, standardize_input=False, include_scale=True
)
Y_offline = jax.vmap(offline_model, in_axes=0)(X)
Y_online = jax.vmap(online_model, in_axes=0)(X)

P_camels = jnp.squeeze(10**Y)
P_offline = jnp.squeeze(10**Y_offline)
P_online = jnp.squeeze(10**Y_online)

N_gas = jax.vmap(cic_paint, in_axes=(None,0))(jnp.zeros(mesh_shape), gas_poss)
gas_N = jax.vmap(cic_read, in_axes=(0, 0))(N_gas, gas_poss)

In [ ]:
plotting.compare_particle_evolution(
    mesh_shape,
    scales,
    jnp.stack([gas_poss, gas_poss, gas_poss], axis=0), 
    jnp.stack([P_camels/gas_N, P_offline/gas_N, P_online/gas_N], axis=0),
    include_pk=True,
    # include_reference=False,
    include_reference=True,
    # values
    shared_colorbar=False,
    individual_colorbars=True,
    # cosmetics
    # title=f"CAMELS (hydro, {CODE})",
    col_titles=["P (CAMELS)", "P (offline fit)", "P (online fit)"],
    # out_dir=f"plots/CAMELS_{CODE}_hydro_evolution_{mesh_per_dim}",
)

In [ ]:
plotting.compare_particle_evolution(
    mesh_shape,
    scales,
    jnp.stack([gas_poss, gas_poss], axis=0), 
    jnp.stack([P_offline/gas_N, P_online/gas_N], axis=0),
    include_pk=True,
    # include_reference=False,
    include_reference=True,
    # values
    shared_colorbar=False,
    individual_colorbars=True,
    # cosmetics
    # title=f"CAMELS (hydro, {CODE})",
    col_titles=["P (offline fit)", "P (online fit)"],
    # out_dir=f"plots/CAMELS_{CODE}_hydro_evolution_{mesh_per_dim}",
)